To implement decoder model architecture, we will need to accomplish following tasks:

1) Text needs to be converted into tokens

2) For step 1, we will need to implement a tokenizer

3) For step 2, we will create a character based tokenizer and train it on data corpus

4) tokenizer should be able to convert text into numbers(encode) and convert numbers back to characters(decode)

5) tokenizer will have its own vocab and dictionaries to perform forward and reverse lookups

6) We will need a dataset which will be converted into iterator and dataloader be created after it. 
Dataloader should create inputs and labels such that labels are 1 position behind the input(DataCollator for Language Modeling in hugging face).

7) We need to create decoder model now

8) First step is to create embedding layer whose input size is equal to vocabulary size and output size is equal to number of dimensions we want to create for each input.

9) Second step is to create positional embeddings whose input size is equal to context length and output isze is equal to model dimension(same as in step 8)

10) add 8+9 and then feed it to decoder block

11) Decoder --> layer norm --> Attention head --> concat attention heads--> add linear layer--> feed forward layer - expand and squeeze with gelu in between (add dropout and layer norm as required)

12) Create decoder layers

13) Finally put linear layer which converts model dimension to vocab dimension for each position

14) train the model 

## Let us implement character tokenizer

Steps in implementing chracter tokenizer:
1) Reading the text and creating a vocabulary
2) Mapping vocab tokens to ids and then from ids to vocab tokens
3) Encode function that takes text; convert it into tokens and then into token ids
4) Decode function that takes tokenids, converts into tokens and then joins tokens to form a coherent text

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
import sys
from torch.utils.data import Dataset, DataLoader, RandomSampler

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


In [3]:
try:
    from datasets import load_dataset
except:
    print("datasets package is not installed. installing it now. Please remember to start kernel once installation is complete")
    !pip install datasets==3.3.2
    print("Please restart your kernel now")

In [4]:
ds = load_dataset("Trelis/tiny-shakespeare")

README.md:   0%|          | 0.00/497 [00:00<?, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/472 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/49 [00:00<?, ? examples/s]

In [5]:
ds

DatasetDict({
    train: Dataset({
        features: ['Text'],
        num_rows: 472
    })
    test: Dataset({
        features: ['Text'],
        num_rows: 49
    })
})

In [6]:
len(ds['train'][0]['Text'])

3131

In [7]:
len_train = len(ds['train'])

In [8]:
for i in range(len_train):
    print(len(ds['train'][i]['Text']))

3131
1703
1614
3199
3159
2872
2969
2847
1930
2451
2507
3046
3122
3081
2701
3265
2478
2750
2796
2550
2863
2410
3173
2218
2813
2562
1264
1708
3239
2945
2417
2488
2554
2601
1802
2767
2743
1029
2009
2165
3295
2541
2435
3013
3128
2843
2930
2428
2502
2964
1997
3116
2622
2794
2849
2636
3142
2763
2656
3038
3090
2896
3239
3070
2438
3158
2773
2050
3261
2886
3234
3076
1406
2902
3065
225
2923
3075
3186
1667
3050
3157
2912
1437
1632
2189
3068
1158
2014
2688
3059
1558
1556
3042
3103
1966
2683
1448
1637
3003
3255
2337
2970
3267
593
2232
2508
3135
3143
2472
2963
3114
3202
3324
3209
3396
2109
2848
2965
3007
2111
2931
3114
3020
3125
2947
2986
3362
2688
1986
2961
1184
2012
3137
2372
2918
2003
3266
1361
1853
2707
3081
3283
3304
2679
3286
1377
1857
3171
3313
3035
3107
1897
3301
3352
3185
1572
1676
2705
3076
2458
3103
2193
3025
2403
1387
1842
3251
3146
905
2399
3387
623
2674
3281
3318
255
3141
3349
3422
3009
2986
3156
3057
3027
3307
2477
2894
62
3238
3228
3113
1558
2635
1339
1676
2985
40
2942
2923
2056
3311

In [9]:
#lets combine all the data together train+test together

In [10]:
ds

DatasetDict({
    train: Dataset({
        features: ['Text'],
        num_rows: 472
    })
    test: Dataset({
        features: ['Text'],
        num_rows: 49
    })
})

In [11]:
df_train = ds['train'].to_pandas()
df_train

,Text
0,"First Citizen:\nBefore we proceed any further,..."
1,"Alack,\nYou are transported by calamity\nThith..."
2,"With a kind of smile,\nWhich ne'er came from t..."
3,"With a kind of smile,\nWhich ne'er came from t..."
4,Your virtue is\nTo make him worthy whose offen...
...,...
467,"To express the like kindness, myself,\nthat ha..."
468,"You are passing welcome,\nAnd so I pray you al..."
469,"You are passing welcome,\nAnd so I pray you al..."
470,"Good morrow, Kate; for that's your name, I hea..."


In [12]:
df_test = ds['test'].to_pandas()
df_test

,Text
0,"TRANIO:\nIs this your speeding? nay, then, goo..."
1,"Sir, list to me:\nI am my father's heir and on..."
2,"And, to cut off all strife, here sit we down:\..."
3,"Now, Licio, to you:\nGood masters, take it not..."
4,"TRANIO:\nPatience, good Katharina, and Baptist..."
5,And yet I come not well.\n\nBAPTISTA:\nAnd yet...
6,"we'll fit him to our turn,--\nAnd he shall be ..."
7,But that his beard grew thin and hungerly\nAnd...
8,"Grumio,\nDraw forth thy weapon, we are beset w..."
9,"But wilt thou make a\nfire, or shall I complai..."


In [13]:
len(df_train), len(df_test)

(472, 49)

In [14]:
df_train.columns, df_test.columns

(Index(['Text'], dtype='object'), Index(['Text'], dtype='object'))

In [15]:
#let us mrge the two datasets together
df = pd.concat([df_train, df_test], axis=0)
df.reset_index(inplace=True, drop=True)
len(df)

521

In [16]:
df

,Text
0,"First Citizen:\nBefore we proceed any further,..."
1,"Alack,\nYou are transported by calamity\nThith..."
2,"With a kind of smile,\nWhich ne'er came from t..."
3,"With a kind of smile,\nWhich ne'er came from t..."
4,Your virtue is\nTo make him worthy whose offen...
...,...
516,What!\nAn advocate for an imposter! hush!\nTho...
517,Our hint of woe\nIs common; every day some sai...
518,The wager?\n\nANTONIO:\nA laughter.\n\nSEBASTI...
519,"GONZALO:\nI assure you, Carthage.\n\nSEBASTIAN..."


In [17]:
shakes_ds = './shakes_ds.csv'
df.to_csv(shakes_ds, index=False)
!ls -ltr

total 169596
drwxrwxr-x  3 ec2-user ec2-user      4096 Feb 22  2025 scripts
-rw-rw-r--  1 ec2-user ec2-user 135751680 Mar 16  2025 brogpt.tar
-rw-rw-r--  1 ec2-user ec2-user  28924927 Mar 22  2025 agnews_train
-rw-rw-r--  1 ec2-user ec2-user   1822708 Mar 22  2025 agnews_test
-rw-rw-r--  1 ec2-user ec2-user    292466 Apr 25 06:48 BroGPT.ipynb
-rw-rw-r--  1 ec2-user ec2-user   1890137 Apr 25 07:35 BroGPT-GPT2TOK.ipynb
drwxrwxr-x  3 ec2-user ec2-user      4096 Apr 25 08:44 model_ckpts
-rw-rw-r--  1 ec2-user ec2-user    129719 Apr 25 11:44 Encoder_MLM.ipynb
-rw-rw-r--  1 ec2-user ec2-user   1341374 Apr 25 15:56 shakes_ds.txt
-rw-rw-r--  1 ec2-user ec2-user   1871240 Apr 25 16:10 BroGPT-GPT2Pretrained.ipynb
drwxrwxr-x 10 ec2-user ec2-user      4096 Apr 26 17:06 encoder-decoder
-rw-rw-r--  1 ec2-user ec2-user    269166 May 21 03:32 BroGPT-v1.ipynb
-rw-rw-r--  1 ec2-user ec2-user   1342940 May 21 03:36 shakes_ds.csv


In [18]:
df['txt_len'] = df['Text'].apply(lambda x: len(x))
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 521 entries, 0 to 520
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Text     521 non-null    object
 1   txt_len  521 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 8.3+ KB


In [19]:
df.describe()

,txt_len
count,521.000000
mean,2574.614203
std,693.507375
min,18.000000
25%,2218.000000
50%,2796.000000
75%,3076.000000
max,3586.000000


In [20]:
#While we can read directly from df/csv to create dataset, we will need to break it as irregular intervals
#as we see there is a huge variation in length of indiviual texts
#let us now write the entire text in normal text file. This will help us batch things better without wasting
#on padding tokens

In [21]:
len(df['Text'].to_list())

521

In [22]:
shakes_txt = './shakes_ds.txt'
with open(shakes_txt, 'w') as f:
    for i in range(len(df)):
        f.write(df['Text'].iloc[i])

In [23]:
!head ./shakes_ds.txt

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:


In [24]:
!tail -50  ./shakes_ds.txt

Execute all things; for no kind of traffic
Would I admit; no name of magistrate;
Letters should not be known; riches, poverty,
And use of service, none; contract, succession,
Bourn, bound of land, tilth, vineyard, none;
No use of metal, corn, or wine, or oil;
No occupation; all men idle, all;
And women too, but innocent and pure;
No sovereignty;--

SEBASTIAN:
Yet he would be king on't.

ANTONIO:
The latter end of his commonwealth forgets the
beginning.

GONZALO:
All things in common nature should produce
Without sweat or endeavour: treason, felony,
Sword, pike, knife, gun, or need of any engine,
Would I not have; but nature should bring forth,
Of its own kind, all foison, all abundance,
To feed my innocent people.

SEBASTIAN:
No marrying 'mong his subjects?

ANTONIO:
None, man; all idle: whores and knaves.

GONZALO:
I would with such perfection govern, sir,
To excel the golden age.

SEBASTIAN:
God save his majesty!

ANTONIO:
Long live Gonzalo!

GONZALO:
And,--do you mark me, sir?

ALON

In [25]:
vocab_text = Path('./shakes_ds.txt').read_text()

In [26]:
#lets try reading from text file in pythonic way
print(vocab_text[0:200])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [27]:
set(vocab_text)

{'\n',
 ' ',
 '!',
 '$',
 '&',
 "'",
 ',',
 '-',
 '.',
 '3',
 ':',
 ';',
 '?',
 'A',
 'B',
 'C',
 'D',
 'E',
 'F',
 'G',
 'H',
 'I',
 'J',
 'K',
 'L',
 'M',
 'N',
 'O',
 'P',
 'Q',
 'R',
 'S',
 'T',
 'U',
 'V',
 'W',
 'X',
 'Y',
 'Z',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z'}

In [28]:
class Char_Tokenizer:
    def __init__(self, text):
        super().__init__()
        self.text = text
        self.vocab = None
        self.tok_list = None
        self.t2i = None
        self.i2t = None
        self.len_vocab = None

    def create_vocab(self):
        self.vocab = set(self.text)
        return sorted(list(self.vocab))

    def train_tokenizer(self):
        self.tok_list = self.create_vocab()
        # tokens created . lets create forwrd and reverse mapping
        self.t2i = {token: index for index, token in enumerate(self.tok_list)}
        self.i2t = {index: token for index, token in enumerate(self.tok_list)}
        self.len_vocab = self.vocab_size()

    def vocab_size(self):
        if self.tok_list is None:
            print("errrrrr Error dude.. you should train the tokenizer first")
        else:
            return len(self.tok_list)

    def encode(self, text, return_tensors=False):
        inputs = [self.t2i[c] for c in text]
        if return_tensors:
            inputs = torch.tensor(inputs)
        return inputs

    def decode(self, inputs, tensors=False):
        if tensors:
            inputs = inputs.squeeze().tolist()
        toks = [self.i2t[i] for i in inputs]
        join_toks = ''.join(toks)
        #right now support is only for one sentence
        return join_toks

    def __str__(self):
        return f"token2index: {self.t2i},\
                index2token: {self.i2t},\
                vocab_len: {self.len_vocab}"

    def __repr__(self):
        return f"token2index: {self.t2i},\
                index2token: {self.i2t},\
                vocab_len: {self.len_vocab}"

In [29]:
tokenizer = Char_Tokenizer(vocab_text)

In [30]:
tokenizer.vocab #Vocab is empty

In [31]:
tokenizer.vocab is None

True

In [32]:
#let us train our tokenizer
tokenizer.train_tokenizer()

In [33]:
tokenizer.vocab is None

False

In [34]:
tokenizer.vocab

{'\n',
 ' ',
 '!',
 '$',
 '&',
 "'",
 ',',
 '-',
 '.',
 '3',
 ':',
 ';',
 '?',
 'A',
 'B',
 'C',
 'D',
 'E',
 'F',
 'G',
 'H',
 'I',
 'J',
 'K',
 'L',
 'M',
 'N',
 'O',
 'P',
 'Q',
 'R',
 'S',
 'T',
 'U',
 'V',
 'W',
 'X',
 'Y',
 'Z',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z'}

In [35]:
tokenizer.i2t

{0: '\n',
 1: ' ',
 2: '!',
 3: '$',
 4: '&',
 5: "'",
 6: ',',
 7: '-',
 8: '.',
 9: '3',
 10: ':',
 11: ';',
 12: '?',
 13: 'A',
 14: 'B',
 15: 'C',
 16: 'D',
 17: 'E',
 18: 'F',
 19: 'G',
 20: 'H',
 21: 'I',
 22: 'J',
 23: 'K',
 24: 'L',
 25: 'M',
 26: 'N',
 27: 'O',
 28: 'P',
 29: 'Q',
 30: 'R',
 31: 'S',
 32: 'T',
 33: 'U',
 34: 'V',
 35: 'W',
 36: 'X',
 37: 'Y',
 38: 'Z',
 39: 'a',
 40: 'b',
 41: 'c',
 42: 'd',
 43: 'e',
 44: 'f',
 45: 'g',
 46: 'h',
 47: 'i',
 48: 'j',
 49: 'k',
 50: 'l',
 51: 'm',
 52: 'n',
 53: 'o',
 54: 'p',
 55: 'q',
 56: 'r',
 57: 's',
 58: 't',
 59: 'u',
 60: 'v',
 61: 'w',
 62: 'x',
 63: 'y',
 64: 'z'}

In [36]:
tokenizer.t2i

{'\n': 0,
 ' ': 1,
 '!': 2,
 '$': 3,
 '&': 4,
 "'": 5,
 ',': 6,
 '-': 7,
 '.': 8,
 '3': 9,
 ':': 10,
 ';': 11,
 '?': 12,
 'A': 13,
 'B': 14,
 'C': 15,
 'D': 16,
 'E': 17,
 'F': 18,
 'G': 19,
 'H': 20,
 'I': 21,
 'J': 22,
 'K': 23,
 'L': 24,
 'M': 25,
 'N': 26,
 'O': 27,
 'P': 28,
 'Q': 29,
 'R': 30,
 'S': 31,
 'T': 32,
 'U': 33,
 'V': 34,
 'W': 35,
 'X': 36,
 'Y': 37,
 'Z': 38,
 'a': 39,
 'b': 40,
 'c': 41,
 'd': 42,
 'e': 43,
 'f': 44,
 'g': 45,
 'h': 46,
 'i': 47,
 'j': 48,
 'k': 49,
 'l': 50,
 'm': 51,
 'n': 52,
 'o': 53,
 'p': 54,
 'q': 55,
 'r': 56,
 's': 57,
 't': 58,
 'u': 59,
 'v': 60,
 'w': 61,
 'x': 62,
 'y': 63,
 'z': 64}

In [37]:
toks = tokenizer.encode("Shaheer")

In [38]:
print(toks)

[31, 46, 39, 46, 43, 43, 56]


In [39]:
tokenizer.decode(toks) 
# Can it really decode me lol!
# Dude dont take it on heart. Its just converting integers to text thats it bro

'Shaheer'

In [40]:
text = "This is a beautiful day"
text

'This is a beautiful day'

In [41]:
enc = tokenizer.encode(text, return_tensors=True)
enc, enc.shape

(tensor([32, 46, 47, 57,  1, 47, 57,  1, 39,  1, 40, 43, 39, 59, 58, 47, 44, 59,
         50,  1, 42, 39, 63]),
 torch.Size([23]))

In [42]:
dec = tokenizer.decode(enc, tensors=True)
dec

'This is a beautiful day'

In [43]:
tokenizer.len_vocab

65

In [44]:
tokenizer

token2index: {'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64},                index2token: {0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43

In [45]:
print(tokenizer)

token2index: {'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64},                index2token: {0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43

In [46]:
class Embed(nn.Module):
    def __init__(self, vocab_size, embed_dim, ctx_len, do, device):
        super().__init__()
        self.tok_layer = nn.Embedding(vocab_size, embed_dim)
        self.pos_layer = nn.Embedding(ctx_len, embed_dim)
        self.norm_do = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(do)
            )
        self.device = device
        self.ctx_len = ctx_len

    def forward(self, inp):
        inp = inp.to(self.device)
        tok_embed = self.tok_layer(inp)
        len_inp = inp.shape[-1]
        try:
            assert len_inp <= self.ctx_len
        except:
            print("Err..Errr. Error...Bro..Length of supplied text exceeds context length defined.Exiting the program now")
            sys.exit(1)
        pos_range = torch.arange(0, len_inp).unsqueeze(0)
        pos_range = pos_range.to(self.device)
        pos_embed = self.pos_layer(pos_range)
        embed_tok_pos = tok_embed + pos_embed
        embed_out = self.norm_do(embed_tok_pos)
        return embed_out

In [47]:
device

'cpu'

In [48]:
embed_mod = Embed(vocab_size=65, embed_dim=768, ctx_len=64, do=0.1, device=device)
embed_mod

Embed(
  (tok_layer): Embedding(65, 768)
  (pos_layer): Embedding(64, 768)
  (norm_do): Sequential(
    (0): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (1): Dropout(p=0.1, inplace=False)
  )
)

In [49]:
#this is just a song that popped in my head from movie named "Amar Akbar Anthony"
text_1 = "My name is Anthony Gonsalvez. Main duniya mein akela hoon."
text_1

'My name is Anthony Gonsalvez. Main duniya mein akela hoon.'

In [50]:
text_2 = "My name is Anthony Gonsalvez. Main duniya mein akela hoon. Dil bhi hai khaali"
text_2, len(text_2)

('My name is Anthony Gonsalvez. Main duniya mein akela hoon. Dil bhi hai khaali',
 77)

In [51]:
toks_1 = tokenizer.encode(text_1, return_tensors=True)
toks_1, toks_1.shape

(tensor([25, 63,  1, 52, 39, 51, 43,  1, 47, 57,  1, 13, 52, 58, 46, 53, 52, 63,
          1, 19, 53, 52, 57, 39, 50, 60, 43, 64,  8,  1, 25, 39, 47, 52,  1, 42,
         59, 52, 47, 63, 39,  1, 51, 43, 47, 52,  1, 39, 49, 43, 50, 39,  1, 46,
         53, 53, 52,  8]),
 torch.Size([58]))

In [52]:
toks_1_embed = embed_mod(toks_1)
toks_1_embed, toks_1_embed.shape

(tensor([[[ 1.4306, -0.8432, -0.2262,  ...,  0.4541, -0.0868,  0.4184],
          [ 0.5810, -0.5322, -0.2585,  ...,  0.5928, -0.8726,  1.8925],
          [ 0.0000, -1.2712, -1.7825,  ...,  0.5603,  0.3001, -1.7606],
          ...,
          [ 1.7750,  1.3187,  0.4860,  ..., -0.0794, -0.6084, -1.2465],
          [-0.2925, -0.3224, -0.7986,  ..., -0.0885,  1.5241, -0.3266],
          [ 0.3488,  0.3298,  0.1780,  ..., -0.6603, -0.5312, -0.0941]]],
        grad_fn=<MulBackward0>),
 torch.Size([1, 58, 768]))

In [53]:
toks_2 = tokenizer.encode(text_2, return_tensors=True)
toks_2, toks_2.shape

(tensor([25, 63,  1, 52, 39, 51, 43,  1, 47, 57,  1, 13, 52, 58, 46, 53, 52, 63,
          1, 19, 53, 52, 57, 39, 50, 60, 43, 64,  8,  1, 25, 39, 47, 52,  1, 42,
         59, 52, 47, 63, 39,  1, 51, 43, 47, 52,  1, 39, 49, 43, 50, 39,  1, 46,
         53, 53, 52,  8,  1, 16, 47, 50,  1, 40, 46, 47,  1, 46, 39, 47,  1, 49,
         46, 39, 39, 50, 47]),
 torch.Size([77]))

In [54]:
#This should error out as tokenized length is greater than context length
toks_2_embed = embed_mod(toks_2)
toks_2_embed, toks_2_embed.shape

Err..Errr. Error...Bro..Length of supplied text exceeds context length defined.Exiting the program now


SystemExit: 1

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [55]:
class Attention(nn.Module):
    def __init__(self, embed_dim, k_dim, do, device):
        super().__init__()
        self.embed_dim = embed_dim
        self.k_dim = k_dim
        self.query = nn.Linear(embed_dim, k_dim)
        self.key = nn.Linear(embed_dim, k_dim)
        self.value = nn.Linear(embed_dim, k_dim)
        self.att_do = nn.Dropout(do)
        self.device = device

    def forward(self, inputs):
        bs = inputs.shape[0]
        num_toks = inputs.shape[1]
        q = self.query(inputs)
        k = self.key(inputs)
        v = self.value(inputs)
        qk = (q@k.transpose(1, 2))/(self.k_dim**0.5)
        t1 = torch.tril(torch.ones(bs, num_toks, num_toks))
        mask = torch.where(t1 == 0, -torch.inf, 0.0)
        mask = mask.to(self.device)
        qk_m = qk + mask
        qk_m_smax = torch.softmax(qk_m, dim=-1)
        qk_m_smax_do = self.att_do(qk_m_smax)
        qkv = qk_m_smax_do@v
        return qkv

In [56]:
att_mod = Attention(embed_dim=768, k_dim=64, do=0.1, device=device)
att_mod

Attention(
  (query): Linear(in_features=768, out_features=64, bias=True)
  (key): Linear(in_features=768, out_features=64, bias=True)
  (value): Linear(in_features=768, out_features=64, bias=True)
  (att_do): Dropout(p=0.1, inplace=False)
)

In [57]:
att_mod_out = att_mod(toks_1_embed)
att_mod_out, att_mod_out.shape

(tensor([[[-0.5319,  0.4052,  0.2823,  ..., -0.4058, -0.7279,  1.0703],
          [ 0.3243,  0.4037,  0.3709,  ..., -0.6666,  0.5148,  0.0898],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          ...,
          [ 0.1628,  0.0562,  0.2255,  ...,  0.0436,  0.2667, -0.0176],
          [ 0.1820,  0.0776,  0.2298,  ..., -0.0915,  0.3680,  0.0294],
          [ 0.1981,  0.0363,  0.2718,  ...,  0.0047,  0.4347, -0.0966]]],
        grad_fn=<UnsafeViewBackward0>),
 torch.Size([1, 58, 64]))

In [58]:
#so things are working good

In [59]:
class Attention_Block(nn.Module):
    def __init__(self, num_heads, embed_dim, k_dim, do, device):
        super().__init__()
        self.num_heads = num_heads
        self.heads_list = [Attention(embed_dim, k_dim, do, device) for i in range(num_heads)]
        self.heads = nn.ModuleList(self.heads_list)
        self.lin_do = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.Dropout(do)
        )

    def forward(self, inputs):
        heads_list_out = [head(inputs) for head in self.heads]
        att_head = torch.cat(heads_list_out, dim=-1)
        att_head_out = self.lin_do(att_head)
        return att_head_out

In [60]:
att_blk = Attention_Block(num_heads=12, embed_dim=768, k_dim=64, do=0.1, device=device)
att_blk

Attention_Block(
  (heads): ModuleList(
    (0-11): 12 x Attention(
      (query): Linear(in_features=768, out_features=64, bias=True)
      (key): Linear(in_features=768, out_features=64, bias=True)
      (value): Linear(in_features=768, out_features=64, bias=True)
      (att_do): Dropout(p=0.1, inplace=False)
    )
  )
  (lin_do): Sequential(
    (0): Linear(in_features=768, out_features=768, bias=True)
    (1): Dropout(p=0.1, inplace=False)
  )
)

In [61]:
att_blk_out = att_blk(toks_1_embed)
att_blk_out, att_blk_out.shape

(tensor([[[-0.0808,  0.4469, -0.0000,  ...,  0.0997, -0.1565, -0.3163],
          [-0.2909,  0.0000,  0.1310,  ...,  0.3251,  0.1751, -0.2239],
          [ 0.2008, -0.1257,  0.0000,  ...,  0.0550, -0.1467, -0.1425],
          ...,
          [-0.0242, -0.1637,  0.2700,  ..., -0.1017, -0.0825,  0.0303],
          [-0.0595, -0.1436,  0.1996,  ..., -0.1212, -0.0000,  0.0317],
          [-0.1042, -0.1791,  0.2302,  ..., -0.0000, -0.1155,  0.0756]]],
        grad_fn=<MulBackward0>),
 torch.Size([1, 58, 768]))

In [62]:
class Transformer_Block(nn.Module):
    def __init__(self, num_heads, embed_dim, k_dim, do, device):
        super().__init__()
        self.MHA = Attention_Block(num_heads, embed_dim, k_dim, do, device)
        self.ff = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim*4),
            nn.GELU(),
            nn.Linear(embed_dim*4, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.Dropout(do)
            )

    def forward(self, inputs):
        inp_start_att_block = inputs
        x = self.MHA(inputs)
        x = x + inp_start_att_block

        inp_start_ff_block = x
        x = self.ff(x)
        x = x + inp_start_ff_block
        return x


In [63]:
trans_blk = Transformer_Block(num_heads=12, embed_dim=768, k_dim=64, do=0.1, device=device)
trans_blk

Transformer_Block(
  (MHA): Attention_Block(
    (heads): ModuleList(
      (0-11): 12 x Attention(
        (query): Linear(in_features=768, out_features=64, bias=True)
        (key): Linear(in_features=768, out_features=64, bias=True)
        (value): Linear(in_features=768, out_features=64, bias=True)
        (att_do): Dropout(p=0.1, inplace=False)
      )
    )
    (lin_do): Sequential(
      (0): Linear(in_features=768, out_features=768, bias=True)
      (1): Dropout(p=0.1, inplace=False)
    )
  )
  (ff): Sequential(
    (0): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=768, out_features=3072, bias=True)
    (2): GELU(approximate='none')
    (3): Linear(in_features=3072, out_features=768, bias=True)
    (4): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (5): Dropout(p=0.1, inplace=False)
  )
)

In [64]:
trans_blk_out = trans_blk(toks_1_embed)
trans_blk_out, trans_blk_out.shape

(tensor([[[ 1.8007, -2.7185,  0.4421,  ..., -1.0782, -0.9855,  0.9289],
          [ 0.0046, -2.5920, -1.1998,  ...,  0.1726, -0.2326,  2.7460],
          [-1.3917,  0.1591, -0.4456,  ..., -0.0085, -0.8037, -3.2188],
          ...,
          [ 2.4030,  0.9573,  0.3795,  ...,  0.2760,  0.0877, -4.0321],
          [-0.2188, -0.3821, -0.6747,  ..., -0.0890,  0.5931,  0.4964],
          [ 1.4304, -0.1049,  2.6136,  ..., -1.4658, -2.4244, -1.7292]]],
        grad_fn=<AddBackward0>),
 torch.Size([1, 58, 768]))

In [65]:
class BroGPT(nn.Module):
    def __init__(self, num_layers, num_heads, vocab_size, embed_dim, k_dim, ctx_len, do, device):
        super().__init__()
        self.emb = Embed(vocab_size, embed_dim, ctx_len, do, device)
        self.layer_list = [Transformer_Block(num_heads, embed_dim, k_dim, do, device) for i in range(num_layers)]
        self.layers = nn.Sequential(*self.layer_list)
        self.embed_vocab = nn.Linear(embed_dim, vocab_size)

    def forward(self, inputs):
        x = self.emb(inputs)
        x = self.layers(x)
        x = self.embed_vocab(x)
        return x

In [66]:
tokenizer.len_vocab

65

In [67]:
num_layers = 6
num_heads = 4
vocab_size = tokenizer.len_vocab
embed_dim = 256
k_dim = 64
ctx_len = 1024
do = 0.1

In [68]:
brogpt = BroGPT(num_layers, num_heads, vocab_size, embed_dim, k_dim, ctx_len, do, device)
brogpt

BroGPT(
  (emb): Embed(
    (tok_layer): Embedding(65, 256)
    (pos_layer): Embedding(1024, 256)
    (norm_do): Sequential(
      (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (1): Dropout(p=0.1, inplace=False)
    )
  )
  (layers): Sequential(
    (0): Transformer_Block(
      (MHA): Attention_Block(
        (heads): ModuleList(
          (0-3): 4 x Attention(
            (query): Linear(in_features=256, out_features=64, bias=True)
            (key): Linear(in_features=256, out_features=64, bias=True)
            (value): Linear(in_features=256, out_features=64, bias=True)
            (att_do): Dropout(p=0.1, inplace=False)
          )
        )
        (lin_do): Sequential(
          (0): Linear(in_features=256, out_features=256, bias=True)
          (1): Dropout(p=0.1, inplace=False)
        )
      )
      (ff): Sequential(
        (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (1): Linear(in_features=256, out_features=1024, bias=True)
      

In [69]:
sum([p.numel() for p in brogpt.parameters()])

5034561

In [70]:
brogpt_out = brogpt(toks_1)
brogpt_out, brogpt_out.shape

(tensor([[[-3.6893,  1.1022,  2.8884,  ...,  4.8646,  2.8707, -0.7234],
          [-0.3063,  0.1199, -0.2190,  ..., -1.5600,  1.7158, -0.6233],
          [-1.4267,  1.8815, -2.1876,  ...,  0.2109, -0.1398, -3.0105],
          ...,
          [ 1.1674, -0.4640, -0.0942,  ...,  1.6564,  1.5582,  0.9019],
          [-0.5147,  0.7005, -0.1355,  ...,  1.2647, -2.5363,  0.3225],
          [-2.1878,  0.7145,  0.8740,  ...,  4.2228, -0.1239, -2.9479]]],
        grad_fn=<ViewBackward0>),
 torch.Size([1, 58, 65]))

In [71]:
# let us test it out again
text = "Its a nice soft morning breeze. I wish I could "
input_ids = tokenizer.encode(text, return_tensors=True)
input_ids, input_ids.shape

(tensor([21, 58, 57,  1, 39,  1, 52, 47, 41, 43,  1, 57, 53, 44, 58,  1, 51, 53,
         56, 52, 47, 52, 45,  1, 40, 56, 43, 43, 64, 43,  8,  1, 21,  1, 61, 47,
         57, 46,  1, 21,  1, 41, 53, 59, 50, 42,  1]),
 torch.Size([47]))

In [72]:
input_ids = input_ids.to(device)
brogpt.to(device)

BroGPT(
  (emb): Embed(
    (tok_layer): Embedding(65, 256)
    (pos_layer): Embedding(1024, 256)
    (norm_do): Sequential(
      (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (1): Dropout(p=0.1, inplace=False)
    )
  )
  (layers): Sequential(
    (0): Transformer_Block(
      (MHA): Attention_Block(
        (heads): ModuleList(
          (0-3): 4 x Attention(
            (query): Linear(in_features=256, out_features=64, bias=True)
            (key): Linear(in_features=256, out_features=64, bias=True)
            (value): Linear(in_features=256, out_features=64, bias=True)
            (att_do): Dropout(p=0.1, inplace=False)
          )
        )
        (lin_do): Sequential(
          (0): Linear(in_features=256, out_features=256, bias=True)
          (1): Dropout(p=0.1, inplace=False)
        )
      )
      (ff): Sequential(
        (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (1): Linear(in_features=256, out_features=1024, bias=True)
      

In [73]:
out = brogpt(input_ids)
out, out.shape

(tensor([[[ 0.6338,  0.5104, -2.6060,  ...,  0.4931,  2.2608, -5.4191],
          [ 0.9668, -2.7311, -3.4659,  ..., -0.1851, -1.3860, -0.8669],
          [-0.5903,  4.3381, -1.7488,  ..., -0.5452, -1.7325,  1.4413],
          ...,
          [ 2.3041, -2.1744, -2.4656,  ..., -1.5775,  1.4606, -0.9726],
          [ 2.3398, -0.9986, -1.6803,  ..., -0.8883, -0.4601,  0.3251],
          [-0.0942,  0.3166, -1.3716,  ..., -0.4516,  1.3988, -1.6015]]],
        grad_fn=<ViewBackward0>),
 torch.Size([1, 47, 65]))

In [74]:
ids = torch.argmax(out, dim=-1)
dec = tokenizer.decode(ids, tensors=True)
print(dec)

gl WRWC.SaWaa.:kr.y3Ar
Wra
DaaWWsWlNmx..WaRlDCQ


In [75]:
len(dec)

47

In [76]:
#lets simulate this in batch format

In [77]:
inp = torch.randint(low=0, high=64, size=(4, 8))
inp = inp.to(device)

In [78]:
inp.shape

torch.Size([4, 8])

In [79]:
out = brogpt(inp)
out, out.shape

(tensor([[[ 7.0003e-01, -4.1778e+00, -2.0065e+00,  ...,  1.8798e+00,
            1.3132e+00,  1.5852e+00],
          [ 3.7273e-01, -4.7624e+00, -1.5738e+00,  ..., -6.9278e-01,
            7.6950e-02,  2.2376e-01],
          [-1.1739e+00, -1.3942e-01, -2.3091e-01,  ...,  3.4995e+00,
           -8.8729e-01,  1.6839e-01],
          ...,
          [-1.4995e+00, -1.0422e+00,  1.2169e+00,  ...,  7.3108e-01,
           -1.9771e+00, -2.5335e+00],
          [-1.8401e+00, -2.0639e+00,  2.0968e+00,  ...,  5.2496e-02,
            1.0045e+00, -7.7444e-01],
          [-1.2866e-01,  1.2120e+00, -2.0230e+00,  ..., -1.2168e+00,
           -8.1566e-01, -3.1295e-01]],
 
         [[-1.1308e+00, -2.2710e+00,  3.9410e-01,  ...,  9.6986e-01,
            1.2013e+00, -1.9438e+00],
          [-1.6328e+00, -2.1161e+00, -1.0897e+00,  ...,  2.4948e+00,
           -3.8506e+00,  8.0914e-01],
          [-6.8692e-02,  1.3553e-01, -1.6297e+00,  ..., -8.5210e-01,
           -8.2406e-01, -1.4418e+00],
          ...,
    

In [80]:
#so this is working in batch format

In [81]:
ids = torch.argmax(out, dim=-1)
ids, ids.shape

(tensor([[57, 45, 34, 39, 19, 40, 37, 46],
         [45, 56,  8, 48, 59, 42,  8, 21],
         [38, 17, 11, 31, 31, 38, 42, 54],
         [57, 50, 34, 61,  7, 47, 45, 35]]),
 torch.Size([4, 8]))

In [82]:
for i in ids:
    dec = tokenizer.decode(i, tensors=True)
    print(dec)

sgVaGbYh
gr.jud.I
ZE;SSZdp
slVw-igW


### Next tasks:
1) Create dataloader
2) Implement training loop and train the model(Use AWS if required)
3) Create a probabilistic decoding mechanism

In [83]:
class Bro_Shake_DS(Dataset):
    def __init__(self, vocab_text, bs=256):
        super().__init__()
        self.data = vocab_text
        self.bs = bs

    def __len__(self):
        return int((len(self.data) - self.bs)/self.bs)

    def __getitem__(self, idx):
        assert idx < len(self.data) - self.bs
        x = tokenizer.encode(self.data[idx:idx+self.bs], return_tensors=True)
        y = tokenizer.encode(self.data[idx+1:idx+1+self.bs], return_tensors=True)

        return x, y

In [84]:
len(vocab_text)

1341374

In [85]:
ds = Bro_Shake_DS(vocab_text[:65536])
ds

In [86]:
len(ds) 

255

In [87]:
dl = DataLoader(ds, shuffle=True, batch_size=32)

In [88]:
len(dl)

8

In [89]:
len(dl.dataset)

255

In [90]:
n=1
for t, l in dl:
    print(f'Sample no: {n}')
    print(f'Text: {t}\n')
    print(f'Label: {l}\n')
    print(t.shape, l.shape)
    n += 1


Sample no: 1
Text: tensor([[43, 44, 53,  ...,  1, 49, 52],
        [31, 54, 43,  ...,  1, 46, 39],
        [41, 43, 43,  ..., 56, 57, 58],
        ...,
        [37, 53, 59,  ..., 57,  5, 58],
        [43,  1, 39,  ..., 60, 43, 56],
        [ 1, 58, 53,  ..., 43,  1, 58]])

Label: tensor([[44, 53, 56,  ..., 49, 52, 53],
        [54, 43, 39,  ..., 46, 39, 60],
        [43, 43, 42,  ..., 57, 58,  1],
        ...,
        [53, 59,  1,  ...,  5, 58,  1],
        [ 1, 39, 50,  ..., 43, 56, 42],
        [58, 53,  1,  ...,  1, 58, 39]])

torch.Size([32, 256]) torch.Size([32, 256])
Sample no: 2
Text: tensor([[43,  8,  0,  ...,  1, 41, 47],
        [ 8,  1, 56,  ..., 39, 61, 39],
        [64, 43, 52,  ..., 41, 43,  8],
        ...,
        [39, 56, 43,  ..., 39,  1, 60],
        [43,  1, 61,  ..., 61,  5, 58],
        [59,  1, 39,  ..., 58,  1, 39]])

Label: tensor([[ 8,  0,  0,  ..., 41, 47, 58],
        [ 1, 56, 43,  ..., 61, 39, 63],
        [43, 52, 10,  ..., 43,  8,  0],
        ...,
      

In [91]:
t, l = t.to(device), l.to(device)
t.shape, l.shape

(torch.Size([31, 256]), torch.Size([31, 256]))

In [92]:
embed = Embed(vocab_size, embed_dim, ctx_len, do, device)
embed.to(device)

Embed(
  (tok_layer): Embedding(65, 256)
  (pos_layer): Embedding(1024, 256)
  (norm_do): Sequential(
    (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (1): Dropout(p=0.1, inplace=False)
  )
)

In [93]:
e_t = embed(t)
e_t.shape

torch.Size([31, 256, 256])

In [94]:
epochs = 1
lr = 0.0006
fn_loss = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(brogpt.parameters(), lr=lr)

In [95]:
sum([p.numel() for p in brogpt.parameters()])

5034561

In [96]:
#we will not train in cpu. we will train in gpu using sagemaker

In [97]:
# TODO: Import any packages that you might need
#importing the "King" library :P
import sagemaker
import boto3
from sagemaker.inputs import TrainingInput

#sagemaker pytorch container estimator related libraries
from sagemaker.pytorch import PyTorch
from sagemaker.pytorch import PyTorchModel


from sagemaker.estimator import Estimator
from sagemaker.model import Model
from sagemaker.predictor import Predictor

#general utility imports
import os
import glob
import pandas as pd
import numpy as np
import random

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [98]:
#Sagemaker related commands
sess = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sess.boto_region_name
bucket = sess.default_bucket()
sess, role, region, bucket

(<sagemaker.session.Session at 0x7f4bae9590c0>,
 'arn:aws:iam::191013407134:role/service-role/AmazonSageMaker-ExecutionRole-20250124T222384',
 'us-east-1',
 'sagemaker-us-east-1-191013407134')

In [99]:
%pwd

'/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/broGPT'

In [100]:
S3_LOC = f's3://{bucket}/gpt_train/dataset/'
S3_LOC

's3://sagemaker-us-east-1-191013407134/gpt_train/dataset/'

In [101]:
#!aws s3 cp ./shakes_ds.txt {S3_LOC}

In [102]:
!aws s3 ls --recursive {S3_LOC}

2025-02-26 05:00:50   27728622 gpt_train/dataset/reuters/reuters_ds.csv
2025-02-20 18:20:57    1341374 gpt_train/dataset/shakes_ds.txt


In [103]:
# let us get image uri
training_image_uri = sagemaker.image_uris.retrieve(
    framework='pytorch', 
    version='2.0',
    instance_type='ml.g4dn.xlarge',
    region=region,
    py_version='py310',
    image_scope='training'
)
print(training_image_uri)

763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-training:2.0-gpu-py310


In [104]:
train_uri = f'{S3_LOC}'
train_uri

's3://sagemaker-us-east-1-191013407134/gpt_train/dataset/'

In [105]:
s3_inp_tr = TrainingInput(
    s3_data = train_uri
    )
s3_inp_tr

In [106]:
data_channels = {
    'train': s3_inp_tr
    }
data_channels

{'train': <sagemaker.inputs.TrainingInput at 0x7f4be02bd600>}

In [107]:
objective_metric_name = "average training loss"
objective_type = "Minimize"
metric_definitions = [{"Name": "average training loss", "Regex": "Average loss: ([0-9\\.]+)"},
                      {"Name": "Perplexity", "Regex": "Perplexity: ([0-9\\.]+)"}]
objective_metric_name, objective_type, metric_definitions

('average training loss',
 'Minimize',
 [{'Name': 'average training loss', 'Regex': 'Average loss: ([0-9\\.]+)'},
  {'Name': 'Perplexity', 'Regex': 'Perplexity: ([0-9\\.]+)'}])

In [108]:
ic=1
i_type = "ml.g4dn.xlarge"
ic, i_type

(1, 'ml.g4dn.xlarge')

In [109]:
%pwd

'/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/broGPT'

In [110]:
hparams = {
    'bs': 128,
    'lrate': 0.0006,
    'num_epochs': 100,
    'do': 0.1,
    'block_size': 256,
    'environ': 'aws',
    'num_layers': 6,
    'num_heads': 4,
    'embed_dim': 256,
    'k_dim': 64,
    'ctx_len': 256
}
hparams

{'bs': 128,
 'lrate': 0.0006,
 'num_epochs': 100,
 'do': 0.1,
 'block_size': 256,
 'environ': 'aws',
 'num_layers': 6,
 'num_heads': 4,
 'embed_dim': 256,
 'k_dim': 64,
 'ctx_len': 256}

In [111]:
est_brogpt = Estimator(
    image_uri=training_image_uri,
    role=role,
    source_dir='./scripts',
    entry_point='train.py',
    instance_count=ic,
    instance_type=i_type,
    base_job_name='brogpt-train',
    hyperparameters=hparams,
    metric_definitions=metric_definitions,
    objective_type=objective_type,
    objective_metric_name=objective_metric_name
)

est_brogpt.fit(
    inputs=data_channels,
    wait=True
)

INFO:sagemaker:Creating training-job with name: brogpt-train-2026-05-21-03-40-03-927


2026-05-21 03:40:06 Starting - Starting the training job...
2026-05-21 03:40:20 Starting - Preparing the instances for training...
2026-05-21 03:40:49 Downloading - Downloading input data...
2026-05-21 03:41:20 Downloading - Downloading the training image.........
2026-05-21 03:43:06 Training - Training image download completed. Training in progress..bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2026-05-21 03:43:08,778 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-05-21 03:43:08,801 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-05-21 03:43:08,812 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2026-05-21 03:43:08,819 sagemaker_pytorch_container.training INFO     Invoking user training script.
2026-05-21 03:43:10,825 sagemaker-training-toolkit INFO     No Neurons detected (normal if no

In [113]:
est_brogpt.model_data

's3://sagemaker-us-east-1-191013407134/brogpt-train-2026-05-21-03-40-03-927/output/model.tar.gz'

In [114]:
est_brogpt.latest_training_job.describe()['FinalMetricDataList']

[{'MetricName': 'Perplexity',
  'Value': 1.0083860158920288,
  'Timestamp': datetime.datetime(2026, 5, 21, 5, 18, 1, tzinfo=tzlocal())},
 {'MetricName': 'average training loss',
  'Value': 0.008351005613803864,
  'Timestamp': datetime.datetime(2026, 5, 21, 5, 18, 1, tzinfo=tzlocal())}]

In [115]:
CKPT_DIR = './model_ckpts'

In [116]:
os.makedirs(CKPT_DIR, exist_ok=True)

In [117]:
!aws s3 cp {est_brogpt.model_data} {CKPT_DIR}/

download: s3://sagemaker-us-east-1-191013407134/brogpt-train-2026-05-21-03-40-03-927/output/model.tar.gz to model_ckpts/model.tar.gz


In [118]:
!aws s3 ls {est_brogpt.model_data}

2026-05-21 05:18:19   53148638 model.tar.gz


In [119]:
%cd {CKPT_DIR}

/home/ec2-user/SageMaker/CURRENTFOCUS/LLMS_transformers_WIP/LLMfromScratch/broGPT/model_ckpts


In [120]:
%pwd
!ls -ltrh

total 2.0G
-rw-r--r-- 1 ec2-user ec2-user  56M Feb 23  2025 ckpt_Epoch_100_Prplxty_1.0083447968191865.pt
-rw-rw-r-- 1 ec2-user ec2-user  51M Feb 23  2025 model_old.tar.gz
-rw-r--r-- 1 ec2-user ec2-user  56M Apr 25 06:31 ckpt_Epoch_50_Prplxty_1.010824581177966.pt
-rw-r--r-- 1 ec2-user ec2-user 351M Apr 25 07:18 ckpt_Epoch_40_Prplxty_1.0619401693860067.pt
-rw-r--r-- 1 ec2-user ec2-user 1.4G Apr 25 08:13 ckpt_Epoch_20_Prplxty_1.0012009449952726.pt
-rw-rw-r-- 1 ec2-user ec2-user  51M May 21 05:18 model.tar.gz


In [121]:
!gunzip -dc model.tar.gz |tar xvf -

tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_100_Prplxty_1.0083859726932272.pt


In [122]:
!ls -ltrh

total 2.0G
-rw-r--r-- 1 ec2-user ec2-user  56M Feb 23  2025 ckpt_Epoch_100_Prplxty_1.0083447968191865.pt
-rw-rw-r-- 1 ec2-user ec2-user  51M Feb 23  2025 model_old.tar.gz
-rw-r--r-- 1 ec2-user ec2-user  56M Apr 25 06:31 ckpt_Epoch_50_Prplxty_1.010824581177966.pt
-rw-r--r-- 1 ec2-user ec2-user 351M Apr 25 07:18 ckpt_Epoch_40_Prplxty_1.0619401693860067.pt
-rw-r--r-- 1 ec2-user ec2-user 1.4G Apr 25 08:13 ckpt_Epoch_20_Prplxty_1.0012009449952726.pt
-rw-r--r-- 1 ec2-user ec2-user  56M May 21 05:18 ckpt_Epoch_100_Prplxty_1.0083859726932272.pt
-rw-rw-r-- 1 ec2-user ec2-user  51M May 21 05:18 model.tar.gz


In [123]:
hparams

{'bs': 128,
 'lrate': 0.0006,
 'num_epochs': 100,
 'do': 0.1,
 'block_size': 256,
 'environ': 'aws',
 'num_layers': 6,
 'num_heads': 4,
 'embed_dim': 256,
 'k_dim': 64,
 'ctx_len': 256}

In [124]:
device

'cpu'

BroGPT(num_layers, num_heads, vocab_size, embed_dim, k_dim, ctx_len, do, device)

In [125]:
model = BroGPT(6, 4, 65, 256, 64, 256, 0.1, device)

In [126]:
with open("./ckpt_Epoch_100_Prplxty_1.0083859726932272.pt", 'rb') as f:
    checkpoint = torch.load(f, map_location=torch.device(device))
    print(f'checkpoint keys: {checkpoint.keys()}')
    model.load_state_dict(checkpoint['model_state_dict'])
    print("completed success")

checkpoint keys: dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'loss', 'device'])
completed success


In [127]:
print(vocab_text[1024:1224])

u proceed especially against Caius Marcius?

All:
Against him first: he's a very dog to the commonalty.

Second Citizen:
Consider you what services he has done for his country?

First Citizen:
Very we


In [128]:
def generate_toks(model, tokenizer, device, prompt, num_toks):
    orig_prompt = prompt
    model.to(device)
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors=True)
    for i in range(num_toks):
        input_ids = input_ids.to(device)
        with torch.no_grad():
            out = model(input_ids)
        ids = torch.argmax(out, dim=-1)
        gen_tok = ids[:,-1]
        input_ids = torch.hstack([input_ids, gen_tok])
    dec = tokenizer.decode(input_ids, tensors=True)
    print(f"Original Prompt: {orig_prompt}")
    print("########################################################")
    print("########################################################")
    print(f"Generated Prompt: {dec}")

In [129]:
prompt = vocab_text[1024:1224]
print(prompt)

u proceed especially against Caius Marcius?

All:
Against him first: he's a very dog to the commonalty.

Second Citizen:
Consider you what services he has done for his country?

First Citizen:
Very we


In [130]:
generate_toks(model, tokenizer, device, prompt, num_toks=57)

Original Prompt: u proceed especially against Caius Marcius?

All:
Against him first: he's a very dog to the commonalty.

Second Citizen:
Consider you what services he has done for his country?

First Citizen:
Very we
########################################################
########################################################
Generated Prompt: u proceed especially against Caius Marcius?

All:
Against him first: he's a very dog to the commonalty.

Second Citizen:
Consider you what services he has done for his country?

First Citizen:
Very well: they are not so be conduct to the crown.

Third Citiz


In [141]:
prompt = vocab_text[1323:1400]
print(prompt)

Second Citizen:
Nay, but speak not maliciously.

First Citizen:
I say unto yo


In [145]:
generate_toks(model, tokenizer, device, prompt, num_toks=178)

Original Prompt: Second Citizen:
Nay, but speak not maliciously.

First Citizen:
I say unto yo
########################################################
########################################################
Generated Prompt: Second Citizen:
Nay, but speak not maliciously.

First Citizen:
I say unto your grace is a man to make of me.

CORIOLANUS:
I would they are a man thou art a man.

Second Citizen:
Ay, ay, ay, if you come to the commonwealth.

Third Citizen:
Now, I will not
